# NB01 — Phase 1: build M_Q and prove it is the same function

**Plan v2 §5, Appendix C. Budget: 3.5 h. Needs a single-GPU 80 GB pod (A100 80 GB or
H100 80 GB) for the 8B pass: the gate holds an unrotated reference and `M_Q` resident at the
same time, ~32 GB of weights plus logits, so a 48 GB pod (A40, L40S, A6000) also works and a
24 GB one does not. The debug path runs on anything.**

The wedge (§2): an orthogonal transform `Q` of `M`'s residual stream leaves an RMSNorm
transformer's function *identical* while sending every extracted activation to `Qv`. So
behavioral similarity between explainer and target is untouched, and coordinate alignment
goes to chance. Anything that survives the rotation is not coordinate-frame.

That argument is only as good as the implementation. This notebook produces `M_Q` and proves
it computes the same function as `M`.

Order matters: fold, verify, *then* rotate. Debugging two transforms at once wastes hours you
don't have.

**What this phase actually buys.** The science only needs `Qv` handed to the explainer, which
is a transform on cached tensors (NB02) and needs no weight surgery at all. What the
weight-level construction buys is the *validity argument*: that `Qv` is a genuine activation
of a real model computing exactly the same function, rather than an arbitrary perturbation.
That argument is worth several hours — but it is an argument, not a pipeline dependency, which
is why v2 §8 lists "Phase 1 on 8B" as cuttable: demonstrate invariance on Qwen3-0.6B, run the
science off the rotated cache, and present 8B invariance as an argument supported by a
smaller-scale check.

Deliverables: `rotate.py` (already in `se/`), `Q` + its seed checkpointed, and the gate
numbers written where the write-up can quote them.


In [ ]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


In [ ]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# The gate holds two 8B bf16 copies at once (~32 GB resident), so 48 GB is the floor
# and 80 GB is comfortable.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=48)

import se_config as C


## 0. Cheap gate first

`se/test_rotate.py` runs the whole construction on a 3-layer model in float64 on CPU: folding
preserves logits, rotation preserves logits, hidden states move by exactly `Q`, tied models
are refused, non-orthogonal maps are refused, and the gate itself can fail. Seconds, and it
catches nearly everything the 8B gate would catch.


In [ ]:
import subprocess

print(subprocess.run([sys.executable, f"{REPO_DIR}/se/test_rotate.py"],
                     capture_output=True, text=True).stdout)


## 0b. Debug on Qwen3-0.6B first

Appendix C.3's practical consequence: 0.6B is a fast substrate for debugging the fold and the
gate, which is where Phase 1's hours actually go. It is tied, so it exercises the untie path
too — which the 8B target never will.

Run this before touching 8B. If it passes here and fails there, the problem is scale or memory,
not algebra, and you are looking in a different place.


In [ ]:
DEBUG_MODEL_ID = "Qwen/Qwen3-0.6B"
RUN_DEBUG_PASS = True

if RUN_DEBUG_PASS:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    import rotate as R

    dbg_tok = AutoTokenizer.from_pretrained(DEBUG_MODEL_ID)
    if dbg_tok.pad_token is None:
        dbg_tok.pad_token = dbg_tok.eos_token

    dbg_ref = AutoModelForCausalLM.from_pretrained(
        DEBUG_MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()
    dbg = AutoModelForCausalLM.from_pretrained(
        DEBUG_MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()

    print(f"{DEBUG_MODEL_ID}: d = {dbg.config.hidden_size}, "
          f"tied = {R.tied_embeddings(dbg)}")
    if R.tied_embeddings(dbg):
        R.untie_embeddings(dbg)
        print("untied lm_head (folding would otherwise corrupt the embedding)")

    R.fold_rmsnorm_gains(dbg)
    dbg_Q = R.random_orthogonal(dbg.config.hidden_size, seed=C.Q_SEED)
    R.apply_rotation(dbg, dbg_Q)

    dbg_texts = ["The capital of France is Paris, and the capital of Germany is Berlin. " * 8,
                 "In a shocking finding, scientists discovered a herd of unicorns. " * 8,
                 "def fibonacci(n):\n    if n < 2:\n        return n\n" * 4,
                 "The mitochondrion is the powerhouse of the cell. " * 8]
    dbg_report = R.invariance_gate(dbg_ref, dbg, dbg_tok, dbg_texts,
                                   max_length=128, batch_size=2)
    print()
    print(R.format_gate_report(dbg_report, f"DEBUG GATE — {DEBUG_MODEL_ID}"))
    assert dbg_report["passed"], "fix the construction here before loading 8B"

    del dbg_ref, dbg
    import gc
    gc.collect()
    torch.cuda.empty_cache()


## 1. Load the target

Unquantized. Appendix C: invariance is exact in float64, close in bfloat16, meaningless in
4-bit.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import rotate as R

MODEL_ID = C.TARGET_MODEL_ID

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
).eval()

d = model.config.hidden_size
print(f"{MODEL_ID}: d = {d}, layers = {model.config.num_hidden_layers}")
print(f"tied embeddings: {R.tied_embeddings(model)}")
assert not R.tied_embeddings(model), "untie first — folding would corrupt the embedding"


## 2. Evaluation texts

~256 FineWeb sequences at 512 tokens (§5.5.1). These are also the texts NB05 fits the ridge
projector on, so they are cached to disk and reused: same distribution, no second download.


In [ ]:
import json

from datasets import load_dataset

GATE_TEXTS_PATH = f"{C.ROTATION_DIR}/gate_texts.json"
N_GATE_SEQS = 256
GATE_MAX_LEN = 512

if os.path.exists(GATE_TEXTS_PATH):
    with open(GATE_TEXTS_PATH) as f:
        gate_texts = json.load(f)
    print(f"loaded {len(gate_texts)} cached gate texts")
else:
    fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT",
                      split="train", streaming=True)
    gate_texts = []
    for row in fw:
        if len(row["text"]) > 2000:            # long enough to fill 512 tokens
            gate_texts.append(row["text"])
        if len(gate_texts) >= N_GATE_SEQS:
            break
    with open(GATE_TEXTS_PATH, "w") as f:
        json.dump(gate_texts, f)
    print(f"cached {len(gate_texts)} gate texts to {GATE_TEXTS_PATH}")


## 3. Fold the RMSNorm gains, and verify folding alone

`RMSNorm(x) = x/rms(x) ⊙ g`. The elementwise gain doesn't commute with `Q`, so absorb it into
every linear that consumes the normalized output:

- `input_layernorm` → `q_proj`, `k_proj`, `v_proj`
- `post_attention_layernorm` → `gate_proj`, `up_proj`
- final `model.norm` → `lm_head`

Qwen3's `q_norm`/`k_norm` are deliberately untouched: they normalize head-dimension slices
downstream of the read matrix, inside the head, so a residual-stream rotation doesn't reach
them. RoPE likewise operates post-projection.

After folding, normalization is pure `x/rms(x)`, which commutes with any orthogonal `Q`.


In [ ]:
import time

t0 = time.time()
folded = R.fold_rmsnorm_gains(model)     # in place
R.assert_no_residual_gains(folded)
print(f"folded in {time.time() - t0:.1f}s; all residual-stream gains are now 1")


In [ ]:
# Gate A: folding alone must not change the function.
# Compare against a freshly loaded unfolded copy.
reference = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
).eval()

fold_report = R.invariance_gate(reference, folded, tokenizer, gate_texts[:64],
                                max_length=GATE_MAX_LEN, batch_size=2)
print(R.format_gate_report(fold_report, "GATE A — folding only (M vs M_folded)"))
assert fold_report["passed"], "folding changed the function; fix this before touching Q"


## 4. Build and apply Q

Haar-uniform `Q ∈ O(d)` via QR of a float64 Gaussian, with the diagonal sign correction that
raw `torch.linalg.qr` omits. Seed fixed and recorded.

Write paths (contribute to the residual) get `W ← QW`; read paths (consume it) get
`W ← WQ^T`; the `[vocab, d]` tensors get `X ← XQ^T`.


In [ ]:
Q = R.random_orthogonal(d, seed=C.Q_SEED)
orth_err = (Q.T @ Q - torch.eye(d, dtype=torch.float64)).abs().max().item()
print(f"Q: {tuple(Q.shape)}, seed = {C.Q_SEED}, max |Q^T Q - I| = {orth_err:.2e}")

q_path = R.save_rotation(Q, C.Q_SEED, f"{C.ROTATION_DIR}/Q_seed{C.Q_SEED}.pt")
print(f"checkpointed to {q_path}")


In [ ]:
t0 = time.time()
rotated = R.apply_rotation(folded, Q)       # in place; `folded` is now M_Q
print(f"rotated in {time.time() - t0:.1f}s")


## 5. Gate B — the invariance check

§5.5: *"Do not proceed without a pass. Highest-value 30 minutes in the project."*

Accept at mean per-token KL ≲ 1e-4 and top-1 agreement ≥ 99.9% in bfloat16. Failures are
nearly always a missed gain fold, a transposed `[vocab, d]` convention, or an unnoticed tied
embedding.


In [ ]:
gate_report = R.invariance_gate(reference, rotated, tokenizer, gate_texts,
                                max_length=GATE_MAX_LEN, batch_size=2)
print(R.format_gate_report(gate_report, "GATE B — full transform (M vs M_Q)"))

gate_report["q_seed"] = C.Q_SEED
gate_report["dtype"] = "bfloat16"
gate_report["model"] = MODEL_ID
with open(f"{C.REPORTS_DIR}/invariance_gate.json", "w") as f:
    json.dump({"fold_only": fold_report, "fold_and_rotate": gate_report}, f, indent=2)
print(f"\nwritten to {C.REPORTS_DIR}/invariance_gate.json — quote these in the write-up (§11.6)")

assert gate_report["passed"], "INVARIANCE GATE FAILED — do not proceed to NB02"


## 6. Second check — labels through the hook path

§5.5: recompute has-changed labels for ~200 patching examples under `M_Q` and confirm they
match the cache. Gate B validates the weights; this validates the transform through the hook
path the experiment actually uses, with `Qv` patched into `M_Q` rather than `v` into `M`.

Field names come from the schema NB00 printed. If `resolve_position` raises, read the message
and add the key — the dataset schema is the only thing here that isn't pinned.


In [ ]:
POSITION_KEYS = ("position", "index", "idx", "token_idx", "token_index", "pos",
                 "orig_text_token_idx", "token_position")


def resolve_position(patch_position):
    for k in POSITION_KEYS:
        if k in patch_position and isinstance(patch_position[k], int):
            return patch_position[k]
    raise KeyError(
        "Could not find the patched token index in patch_position. "
        f"Available keys: {sorted(patch_position.keys())}. "
        "Add the right one to POSITION_KEYS."
    )


def patch_hook(vector, position):
    """Replace the residual stream at `position` with `vector`, at each hooked layer."""
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        hidden[:, position, :] = vector.to(hidden.dtype)
        return (hidden,) + output[1:] if isinstance(output, tuple) else hidden
    return hook


@torch.no_grad()
def replay_patch(model, example, vector, n_new_tokens):
    """Run the cached patch through `model` and return the greedy continuation."""
    layers = example["layer"]
    position = resolve_position(example["patch_position"])
    input_ids = torch.tensor(
        [tokenizer.convert_tokens_to_ids(example["input_tokens"])], device=model.device
    )
    v = torch.tensor(vector, device=model.device)

    handles = [model.model.layers[l].register_forward_hook(patch_hook(v, position))
               for l in layers]
    try:
        out = model.generate(input_ids, max_new_tokens=n_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    finally:
        for h in handles:
            h.remove()
    return tokenizer.convert_ids_to_tokens(out[0, input_ids.shape[1]:])


In [ ]:
N_LABEL_CHECK = 200

label_ds = load_dataset(C.ACT_DATASET, split=f"train[:{N_LABEL_CHECK}]")
agree, checked, failures = 0, 0, []

for i, ex in enumerate(label_ds):
    try:
        v = torch.tensor(ex["patch_position"]["intervention_vector"], dtype=torch.float64)
        Qv = (Q @ v).to(torch.bfloat16)
        n_new = len(ex["ablated_continuation"])
        got = replay_patch(rotated, ex, Qv, n_new)
    except KeyError as err:
        print(err)
        break
    except Exception as err:                       # noqa: BLE001 — log and continue
        failures.append((i, repr(err)))
        continue

    cached = ex["ablated_continuation"]
    checked += 1
    agree += int("".join(got) == "".join(cached))

if checked:
    print(f"replayed {checked} examples under M_Q with Qv patched in")
    print(f"  continuation matches cache : {agree}/{checked} = {agree/checked:.3f}")
    print(f"  errors                     : {len(failures)}")
    with open(f"{C.REPORTS_DIR}/label_recheck.json", "w") as f:
        json.dump({"checked": checked, "agree": agree, "rate": agree / checked,
                   "errors": failures[:20]}, f, indent=2)


**Reading this check.** Exact continuation agreement will not be 100%, and shouldn't be
expected to: greedy decoding in bfloat16 through a rotated model accumulates different
rounding, and a near-tie at any position flips the whole suffix. What matters is that
agreement is *high* (well above 90%) and that disagreements are near-ties rather than
different predictions. A low rate means the transform is wrong somewhere the logit gate
didn't reach.

If it is low and Gate B passed, suspect `resolve_position` picking the wrong field before you
suspect the rotation.


## 7. What to carry forward

`M_Q` is **not** saved. Nothing downstream needs it: training and evaluation consume cached
activations, and the only thing that has to persist is `Q` itself, which is 134 MB in float64
and reproducible from its seed. Saving 16 GB of rotated weights to the volume would cost more
time than rebuilding them — and more than the pod-hours it would save.

Next: **NB02** applies `v ↦ Qv` to the cached activation dataset.


In [ ]:
print(f"Q            : {q_path}  (seed {C.Q_SEED})")
print(f"gate report  : {C.REPORTS_DIR}/invariance_gate.json")
print(f"gate passed  : {gate_report['passed']}")
print(f"gate texts   : {GATE_TEXTS_PATH}  (reused by NB05's ridge fit)")

del reference, rotated, folded, model
import gc
gc.collect()
torch.cuda.empty_cache()
